In [1]:
import pandas as pd
data = pd.read_csv(r"C:\Users\Admin\Downloads\IoT Dataset\CCIOT\combined_dataset.csv")

In [2]:
data.columns

Index(['device_name', 'device_mac', 'label_full', 'label1', 'label2', 'label3',
       'label4', 'timestamp', 'timestamp_start', 'timestamp_end',
       'log_data-ranges_avg', 'log_data-ranges_max', 'log_data-ranges_min',
       'log_data-ranges_std_deviation', 'log_data-types',
       'log_data-types_count', 'log_interval-messages', 'log_messages_count',
       'network_fragmentation-score', 'network_fragmented-packets',
       'network_header-length_avg', 'network_header-length_max',
       'network_header-length_min', 'network_header-length_std_deviation',
       'network_interval-packets', 'network_ip-flags_avg',
       'network_ip-flags_max', 'network_ip-flags_min',
       'network_ip-flags_std_deviation', 'network_ip-length_avg',
       'network_ip-length_max', 'network_ip-length_min',
       'network_ip-length_std_deviation', 'network_ips_all',
       'network_ips_all_count', 'network_ips_dst', 'network_ips_dst_count',
       'network_ips_src', 'network_ips_src_count', 'network_

In [3]:
print(data["label2"].value_counts())

label2
benign        400672
recon         105848
dos            57736
ddos           56692
mitm           25490
malware        24177
web             9040
bruteforce      6016
Name: count, dtype: int64


In [4]:
start_dt = pd.to_datetime(data["timestamp_start"])
end_dt = pd.to_datetime(data["timestamp_end"])
duration = (end_dt - start_dt).dt.total_seconds()
data_1s = data[duration == 1.0].reset_index(drop=True)

print(data_1s["label2"].value_counts())

label2
benign        136800
recon          33648
dos            18420
ddos           18056
mitm            8062
malware         7541
web             2796
bruteforce      1868
Name: count, dtype: int64


In [5]:
print(data_1s["timestamp_start"].head())

0    2025-01-23T15:31:10.709000Z
1    2025-01-23T15:31:11.709000Z
2    2025-01-23T15:31:12.709000Z
3    2025-01-23T15:31:13.709000Z
4    2025-01-23T15:31:14.709000Z
Name: timestamp_start, dtype: object


In [6]:
data_1s.sort_values("timestamp_start", inplace=True)

In [7]:
# lưu data_1s đã được lọc và sắp xếp (định dạng parquet)
data_1s.to_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\CCIOT\saved_preprocessed\data_1s.parquet", index=False)

One-hot encoding

In [8]:
cols_to_drop = ["device_name","device_mac","timestamp","timestamp_end","network_ips_all","network_macs_all","network_macs_dst","network_macs_src","network_ports_all","network_protocols_all","label1", "label3", "label4", "label_full"]
data_1s.drop(columns = cols_to_drop, inplace=True)

In [9]:
import joblib
saved_folder = r"C:\Users\Admin\Downloads\IoT Dataset\CCIOT\saved_preprocessed"
joblib.dump(cols_to_drop, f"{saved_folder}/cols_to_drop.pkl")

['C:\\Users\\Admin\\Downloads\\IoT Dataset\\CCIOT\\saved_preprocessed/cols_to_drop.pkl']

In [10]:
import pandas as pd
import ast
from sklearn.preprocessing import MultiLabelBinarizer
import joblib

# ==========================================
# PIPELINE XỬ LÝ
# ==========================================
saved_folder = r"C:\Users\Admin\Downloads\IoT Dataset\CCIOT\saved_preprocessed"
# Bước 1: Parse (Chuyển đổi) chuỗi thành List thực sự
def parse_string_to_list(val):
    try:
        # ast.literal_eval dịch chuỗi "['a', 'b']" thành cấu trúc list ['a', 'b'] của Python
        if isinstance(val, str):
            return ast.literal_eval(val)
        return val if isinstance(val, list) else []
    except (ValueError, SyntaxError):
        # Trả về list rỗng nếu gặp giá trị lỗi/NaN
        return []

# Áp dụng hàm parse tạo một cột tạm
data_1s['parsed_list'] = data_1s['log_data-types'].apply(parse_string_to_list)


# Bước 2: Khởi tạo và áp dụng MultiLabelBinarizer
mlb = MultiLabelBinarizer()
# fit_transform sẽ quét qua các list và tạo ra ma trận nhị phân (0 và 1)
binary_matrix = mlb.fit_transform(data_1s['parsed_list'])
joblib.dump(mlb, f"{saved_folder}/mlb_log_data_types.pkl")
# Lấy tên các nhãn (classes) mà mô hình tìm thấy (array, numeric, string)
# Thêm tiền tố 'log_type_' để tên cột rõ nghĩa hơn khi đưa vào mô hình học máy
new_column_names = [f"log_type_{c}" for c in mlb.classes_]

# Biến ma trận thành DataFrame
df_binary = pd.DataFrame(binary_matrix, columns=new_column_names, index=data_1s.index)


# Bước 3: Nối dữ liệu mới vào bảng gốc và dọn dẹp
df_final = pd.concat([data_1s, df_binary], axis=1)

# Xóa bỏ cột chuỗi gốc và cột list tạm thời
df_final = df_final.drop(columns=['log_data-types', 'parsed_list'])

print("--- DỮ LIỆU SAU KHI XỬ LÝ ---")
print(df_final)

--- DỮ LIỆU SAU KHI XỬ LÝ ---
        label2              timestamp_start  log_data-ranges_avg  \
63569    recon  2025-01-15T13:04:54.353000Z                  0.0   
63969    recon  2025-01-15T13:04:54.369000Z                  0.0   
62108    recon  2025-01-15T13:04:54.412000Z                  0.0   
62447    recon  2025-01-15T13:04:54.412000Z               1023.0   
62048    recon  2025-01-15T13:04:54.539000Z                  0.0   
...        ...                          ...                  ...   
223590  benign  2025-09-09T15:09:39.400000Z                  0.0   
180390  benign  2025-09-09T15:09:39.400000Z                  0.0   
126390  benign  2025-09-09T15:09:39.400000Z                211.0   
187590  benign  2025-09-09T15:09:39.400000Z                  0.0   
227190  benign  2025-09-09T15:09:39.400000Z                  0.0   

        log_data-ranges_max  log_data-ranges_min  \
63569                   0.0                  0.0   
63969                   0.0                  0.0 

In [11]:
df_final.sort_values("timestamp_start", inplace=True)

In [12]:
import joblib
def process_protocol_column(df, col_name, prefix='proto', threshold=100):
    print(f"Bắt đầu xử lý cột: {col_name}")
    # Parse chuỗi thành list (sử dụng hàm parse_string_to_list bạn đã định nghĩa ở cell trước)
    parsed_series = df[col_name].apply(parse_string_to_list)
    
    # Đếm tần suất xuất hiện của từng phần tử trong các list
    all_elements = [item for sublist in parsed_series for item in sublist]
    counts = pd.Series(all_elements).value_counts()
    
    # Tập hợp các giao thức xuất hiện > threshold
    frequent_protos = set(counts[counts > threshold].index)
    print(f" - Số giao thức > {threshold} lần: {len(frequent_protos)}")
    
    # Hàm gom nhóm
    def group_protocols(proto_list):
        res = set()
        for p in proto_list:
            if p in frequent_protos:
                res.add(p)
            else:
                res.add('other')
        return list(res)
        
    # Áp dụng hàm gom nhóm
    grouped_series = parsed_series.apply(group_protocols)
    
    # Sử dụng MultiLabelBinarizer
    mlb = MultiLabelBinarizer()
    binary_matrix = mlb.fit_transform(grouped_series)

    # Tạo DataFrame mới với tên cột thêm prefix
    new_cols = [f"{prefix}_{c}" for c in mlb.classes_]
    df_binary = pd.DataFrame(binary_matrix, columns=new_cols, index=df.index)
    
    # Nối vào bảng gốc và Drop cột dữ liệu chuỗi ban đầu đi
    df_out = pd.concat([df, df_binary], axis=1).drop(columns=[col_name])
    print(f" - Hoàn tất! Tạo ra {len(new_cols)} cột mới: {new_cols[:5]} ...")
    return df_out, frequent_protos,mlb

# ================================
# ÁP DỤNG HÀM VÀO CÁC CỘT CỦA BẠN
# ================================
saved_folder = r"C:\Users\Admin\Downloads\IoT Dataset\CCIOT\saved_preprocessed"
if 'network_protocols_src' in df_final.columns:
    df_final, freq_src, mlb_src = process_protocol_column(df_final, 'network_protocols_src', prefix='src_proto', threshold=100)
    joblib.dump(mlb_src, f"{saved_folder}/mlb_network_protocols_src.pkl")
    joblib.dump(freq_src, f"{saved_folder}/freq_network_protocols_src.pkl")

if 'network_protocols_dst' in df_final.columns:
    df_final, freq_dst, mlb_dst = process_protocol_column(df_final, 'network_protocols_dst', prefix='dst_proto', threshold=100)
    joblib.dump(mlb_dst, f"{saved_folder}/mlb_network_protocols_dst.pkl")
    joblib.dump(freq_dst, f"{saved_folder}/freq_network_protocols_dst.pkl")

print("\n--- Kiểm tra lại các cột Object còn sót lại ---")
print(df_final.select_dtypes(include=['object']).columns)
print("\nKích thước df_final mới:", df_final.shape)

Bắt đầu xử lý cột: network_protocols_src
 - Số giao thức > 100 lần: 22
 - Hoàn tất! Tạo ra 23 cột mới: ['src_proto_arp', 'src_proto_data', 'src_proto_data-text-lines', 'src_proto_dhcpv6', 'src_proto_dns'] ...
Bắt đầu xử lý cột: network_protocols_dst
 - Số giao thức > 100 lần: 35
 - Hoàn tất! Tạo ra 36 cột mới: ['dst_proto_arp', 'dst_proto_c1222', 'dst_proto_chargen', 'dst_proto_data', 'dst_proto_data-text-lines'] ...

--- Kiểm tra lại các cột Object còn sót lại ---
Index(['label2', 'timestamp_start', 'network_ips_dst', 'network_ips_src',
       'network_ports_dst', 'network_ports_src'],
      dtype='object')

Kích thước df_final mới: (227191, 139)


In [13]:
df_final.rename(columns={"timestamp_start": "timestamp", "label2": "label"}, inplace=True)

In [14]:
# chia ra 3 tập train, val, test theo stratify label (và trong mỗi label, lấy theo tỷ lệ 70:15:15 chia theo timestamp để đảm bảo tính liên tục của dữ liệu thời gian)
# Nghĩa là đối với mỗi class 70% sample có timestamp nhỏ nhất được lấy làm train, 15 tiếp theo là valid, còn lại là test.

train_list = []
val_list = []
test_list = []

# Đảm bảo dữ liệu đã được sắp xếp theo thời gian
df_final = df_final.sort_values(by='timestamp')

for label_val, group in df_final.groupby('label'):
    n = len(group)
    n_train = int(n * 0.70)
    n_val = int(n * 0.15)
    
    train_group = group.iloc[:n_train]
    val_group = group.iloc[n_train:n_train + n_val]
    test_group = group.iloc[n_train + n_val:]
    
    train_list.append(train_group)
    val_list.append(val_group)
    test_list.append(test_group)

df_train = pd.concat(train_list).sort_values(by='timestamp')
df_val = pd.concat(val_list).sort_values(by='timestamp')
df_test = pd.concat(test_list).sort_values(by='timestamp')

# Tách features và target
X_train = df_train.drop(columns=['label', 'timestamp'])
y_train = df_train['label']

X_val = df_val.drop(columns=['label', 'timestamp'])
y_val = df_val['label']

X_test = df_test.drop(columns=['label', 'timestamp'])
y_test = df_test['label']

print("Train shape:", X_train.shape)
print("Val shape:", X_val.shape)
print("Test shape:", X_test.shape)

Train shape: (159031, 137)
Val shape: (34077, 137)
Test shape: (34083, 137)


In [15]:
# kết hợp train_df, valid_df, test_df lại (gộp cột x và y lại thành 1 bảng duy nhất)
train_df = pd.concat([X_train, y_train], axis=1)
valid_df = pd.concat([X_val, y_val], axis=1)
test_df = pd.concat([X_test, y_test], axis=1)

In [16]:
# áp dụng quantile normalization cho các cột số
from sklearn.preprocessing import QuantileTransformer
scaler = QuantileTransformer(output_distribution='normal', random_state=42)

# Lấy danh sách các cột số (loại bỏ cột label nếu nó vô tình bị hiểu là số)
numeric_cols = train_df.select_dtypes(include=['number']).columns.tolist()
if 'label' in numeric_cols:
    numeric_cols.remove('label')

# áp dụng scaler cho các cột số trong train, val, test
train_df[numeric_cols] = scaler.fit_transform(train_df[numeric_cols])
valid_df[numeric_cols] = scaler.transform(valid_df[numeric_cols])
test_df[numeric_cols] = scaler.transform(test_df[numeric_cols])

# áp dụng label encoding cho cột label
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
train_df['label'] = le.fit_transform(train_df['label'])
valid_df['label'] = le.transform(valid_df['label'])
test_df['label'] = le.transform(test_df['label'])


joblib.dump(scaler, f"{saved_folder}/quantile_scaler.pkl")
joblib.dump(numeric_cols, f"{saved_folder}/numeric_columns.pkl")
joblib.dump(le, f"{saved_folder}/label_encoder.pkl")

['C:\\Users\\Admin\\Downloads\\IoT Dataset\\CCIOT\\saved_preprocessed/label_encoder.pkl']

In [17]:
label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print("Label mapping:", label_mapping)

Label mapping: {'benign': 0, 'bruteforce': 1, 'ddos': 2, 'dos': 3, 'malware': 4, 'mitm': 5, 'recon': 6, 'web': 7}


In [19]:
def generate_spark_schema_code(df):
    """
    Hàm tự động đọc dtypes của Pandas DataFrame và sinh ra 
    đoạn code Python khai báo StructType cho PySpark.
    """
    print("from pyspark.sql.types import StructType, StructField, FloatType, IntegerType, StringType, BooleanType\n")
    print("output_schema = StructType([")
    
    for col_name, dtype in df.dtypes.items():
        dtype_str = str(dtype).lower()
        
        # Quyết định kiểu dữ liệu Spark dựa trên Pandas dtype
        if 'float' in dtype_str:
            spark_type = "FloatType()"
        elif 'int' in dtype_str:
            spark_type = "IntegerType()"
        elif 'bool' in dtype_str:
            spark_type = "BooleanType()"
        else:
            spark_type = "StringType()"
            
        # In ra từng dòng StructField
        print(f'    StructField("{col_name}", {spark_type}, True),')
        
    print("])")

# Gọi hàm với train_df của bạn
generate_spark_schema_code(train_df)

from pyspark.sql.types import StructType, StructField, FloatType, IntegerType, StringType, BooleanType

output_schema = StructType([
    StructField("log_data-ranges_avg", FloatType(), True),
    StructField("log_data-ranges_max", FloatType(), True),
    StructField("log_data-ranges_min", FloatType(), True),
    StructField("log_data-ranges_std_deviation", FloatType(), True),
    StructField("log_data-types_count", FloatType(), True),
    StructField("log_interval-messages", FloatType(), True),
    StructField("log_messages_count", FloatType(), True),
    StructField("network_fragmentation-score", FloatType(), True),
    StructField("network_fragmented-packets", FloatType(), True),
    StructField("network_header-length_avg", FloatType(), True),
    StructField("network_header-length_max", FloatType(), True),
    StructField("network_header-length_min", FloatType(), True),
    StructField("network_header-length_std_deviation", FloatType(), True),
    StructField("network_interval-packe

In [23]:
import pandas as pd

# 1. Đọc file thực tế của bạn
df_actual = pd.read_csv(r"C:\Users\Admin\Downloads\IoT Dataset\CCIOT\train_final.csv", nrows=100) # Đọc 100 dòng cho nhanh

# 2. Chạy hàm sinh code Spark Schema
def generate_spark_schema_code(df):
    print("from pyspark.sql.types import StructType, StructField, FloatType, IntegerType, StringType, BooleanType\n")
    print("output_schema = StructType([")
    for col_name, dtype in df.dtypes.items():
        dtype_str = str(dtype).lower()
        if 'float' in dtype_str: spark_type = "FloatType()"
        elif 'int' in dtype_str: spark_type = "IntegerType()"
        elif 'bool' in dtype_str: spark_type = "BooleanType()"
        else: spark_type = "StringType()"
        print(f'    StructField("{col_name}", {spark_type}, True),')
    print("])")

generate_spark_schema_code(df_actual)

from pyspark.sql.types import StructType, StructField, FloatType, IntegerType, StringType, BooleanType

output_schema = StructType([
    StructField("label", IntegerType(), True),
    StructField("timestamp", StringType(), True),
    StructField("log_data-ranges_avg", FloatType(), True),
    StructField("log_data-ranges_max", FloatType(), True),
    StructField("log_data-ranges_min", FloatType(), True),
    StructField("log_data-ranges_std_deviation", FloatType(), True),
    StructField("log_data-types_count", FloatType(), True),
    StructField("log_interval-messages", FloatType(), True),
    StructField("log_messages_count", FloatType(), True),
    StructField("network_fragmentation-score", FloatType(), True),
    StructField("network_fragmented-packets", FloatType(), True),
    StructField("network_header-length_avg", FloatType(), True),
    StructField("network_header-length_max", FloatType(), True),
    StructField("network_header-length_min", FloatType(), True),
    StructField("

In [25]:
df_input = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\CCIOT\saved_preprocessed\data_1s.parquet",engine='pyarrow')
generate_spark_schema_code(df_input)

from pyspark.sql.types import StructType, StructField, FloatType, IntegerType, StringType, BooleanType

output_schema = StructType([
    StructField("device_name", StringType(), True),
    StructField("device_mac", StringType(), True),
    StructField("label_full", StringType(), True),
    StructField("label1", StringType(), True),
    StructField("label2", StringType(), True),
    StructField("label3", StringType(), True),
    StructField("label4", StringType(), True),
    StructField("timestamp", StringType(), True),
    StructField("timestamp_start", StringType(), True),
    StructField("timestamp_end", StringType(), True),
    StructField("log_data-ranges_avg", FloatType(), True),
    StructField("log_data-ranges_max", FloatType(), True),
    StructField("log_data-ranges_min", FloatType(), True),
    StructField("log_data-ranges_std_deviation", FloatType(), True),
    StructField("log_data-types", StringType(), True),
    StructField("log_data-types_count", IntegerType(), True),
  